# 02 Exploratory Analysis

This notebook explores the synthetic Formula 1 race dataset to understand lap time trends, tyre compound behaviour, tyre degradation, safety car impact and team pace differences.

The aim is to identify the main factors that influence race pace before building a lap time prediction model.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

os.makedirs("../visuals", exist_ok=True)

In [ ]:
df = pd.read_csv("../data/raw/synthetic_f1_race_data.csv")

df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

## Lap Time Trend by Driver

This chart shows how lap times change across the race for each driver.

Safety car laps are expected to appear as a clear spike in lap time.

In [ ]:
plt.figure(figsize=(12, 6))

for driver in df["driver"].unique():
    driver_data = df[df["driver"] == driver]
    plt.plot(driver_data["lap"], driver_data["lap_time_seconds"], label=driver)

plt.title("Lap Time Trend by Driver")
plt.xlabel("Lap")
plt.ylabel("Lap Time (seconds)")
plt.legend()
plt.tight_layout()
plt.savefig("../visuals/lap_time_trend_by_driver.png", bbox_inches="tight")
plt.show()

## Average Lap Time by Team

This analysis compares average lap time by team.

Safety car laps are excluded to give a cleaner comparison of race pace.

In [ ]:
green_flag_df = df[df["safety_car"] == 0]

team_pace = (
    green_flag_df.groupby("team")["lap_time_seconds"]
    .mean()
    .sort_values()
)

team_pace

In [ ]:
team_pace.plot(kind="bar")

plt.title("Average Green Flag Lap Time by Team")
plt.xlabel("Team")
plt.ylabel("Average Lap Time (seconds)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("../visuals/average_lap_time_by_team.png", bbox_inches="tight")
plt.show()

## Tyre Compound Performance

This section compares average lap time across tyre compounds.

Safety car laps are excluded because they artificially increase lap times.

In [ ]:
compound_pace = (
    green_flag_df.groupby("compound")["lap_time_seconds"]
    .mean()
    .sort_values()
)

compound_pace

In [ ]:
compound_pace.plot(kind="bar")

plt.title("Average Green Flag Lap Time by Tyre Compound")
plt.xlabel("Tyre Compound")
plt.ylabel("Average Lap Time (seconds)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("../visuals/average_lap_time_by_compound.png", bbox_inches="tight")
plt.show()

## Tyre Degradation by Compound

This chart shows how lap time changes as tyre age increases for each compound.

Soft tyres should generally show stronger degradation, while hard tyres should degrade more slowly.

In [ ]:
degradation = (
    green_flag_df.groupby(["compound", "tyre_age"])["lap_time_seconds"]
    .mean()
    .reset_index()
)

degradation.head()

In [ ]:
plt.figure(figsize=(10, 6))

for compound in degradation["compound"].unique():
    compound_data = degradation[degradation["compound"] == compound]
    plt.plot(
        compound_data["tyre_age"],
        compound_data["lap_time_seconds"],
        marker="o",
        label=compound
    )

plt.title("Tyre Degradation by Compound")
plt.xlabel("Tyre Age (laps)")
plt.ylabel("Average Lap Time (seconds)")
plt.legend()
plt.tight_layout()
plt.savefig("../visuals/tyre_degradation_by_compound.png", bbox_inches="tight")
plt.show()

## Safety Car Impact

Safety car laps significantly increase lap times and can change race strategy decisions.

This section compares normal racing laps against safety car laps.

In [ ]:
safety_car_impact = (
    df.groupby("safety_car")["lap_time_seconds"]
    .mean()
)

safety_car_impact

In [ ]:
safety_car_impact.index = ["Green Flag", "Safety Car"]

safety_car_impact.plot(kind="bar")

plt.title("Average Lap Time: Green Flag vs Safety Car")
plt.xlabel("Race Condition")
plt.ylabel("Average Lap Time (seconds)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("../visuals/safety_car_impact.png", bbox_inches="tight")
plt.show()

## Mercedes Race Pace Analysis

Since this project is inspired by a Mercedes-style race strategy team, this section focuses on Mercedes driver race pace.

In [ ]:
mercedes_df = green_flag_df[green_flag_df["team"] == "Mercedes"]

mercedes_df.groupby("driver")["lap_time_seconds"].mean()

In [ ]:
plt.figure(figsize=(10, 5))

for driver in mercedes_df["driver"].unique():
    driver_data = mercedes_df[mercedes_df["driver"] == driver]
    plt.plot(driver_data["lap"], driver_data["lap_time_seconds"], label=driver)

plt.title("Mercedes Green Flag Lap Time Trend")
plt.xlabel("Lap")
plt.ylabel("Lap Time (seconds)")
plt.legend()
plt.tight_layout()
plt.savefig("../visuals/mercedes_lap_time_trend.png", bbox_inches="tight")
plt.show()

## Key Findings

Based on the exploratory analysis:

- Lap times improve during the race as fuel load decreases.
- Tyre age increases lap time due to degradation.
- Soft tyres are faster initially but degrade more quickly.
- Hard tyres are slower initially but more stable over longer stints.
- Safety car laps create major lap time spikes and need to be treated separately in modelling.
- Team and driver pace differences are important inputs for lap time prediction.

These findings will guide the feature selection in the lap time prediction model.